# Minimal cartpole reachability tutorial

This notebook provides a minimal example that reproduces the core reachability path of
`python run_ctl.py config/ct_ctl/cartpole.yaml --sim --ver`, but configures the
cartpole example directly in Python.

For your own controlled continuous-time system, replace the dimensions, paths,
initial box, and timing parameters below. The dynamics file should expose a
`dynamics(x)` function, and the controller should be an ONNX model accepted by
`load_controller`.

In [8]:
from pathlib import Path
import time

repo_root = Path.cwd()

import jax
# CPU usually compiles faster and is convenient for a tutorial.
# Comment out the next line to run on GPU.
jax.config.update("jax_platform_name", "cpu")

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")
import jax.numpy as jnp

from src.reachability import CT_Ctl_Reach
from src.utils.box_set import prepare_initial_sets
from src.utils.load import load_analytic_dynamics, load_controller

## 1. Configure the problem in Python

In [9]:
# System dimensions and variable names
state_dim = 4
action_dim = 1
total_dim = state_dim + action_dim
variable_names = ["x1", "x2", "x3", "x4", "u1"]

# Time parameters
n_total_steps = 200
n_steps_per_control = 4
step_size = 0.005

# Reachability parameters
init_remainder = jnp.array([1e-5])
frr_rounds = 10
frr_stop_ratio = 0.95
sr_window_size = 1000

# Paths to the dynamics and controller files
dynamics_path = repo_root / "models/dynamics/ct_ctl/cartpole.py"
controller_path = repo_root / "models/controllers/ct_ctl/cartpole.onnx"

# Initial set
initial_set = jnp.array(
    [
        [-0.0375,  -0.03125],  # x1
        [-0.015625, -0.0125],  # x2
        [-0.00625,   0.0],     # x3
        [-0.007375, -0.00625], # x4
        [0.0,        0.0],     # u1, overwritten by the controller
    ],
    dtype=jnp.float32,
)

# Split only the state dimensions. The action dimension stays fixed initially.
splits = {0: 8, 1: 8, 2: 8, 3: 8}

## 2. Load the dynamics, controller, and initial boxes

In [10]:
rhs = load_analytic_dynamics(
    str(dynamics_path),
    D_s=state_dim,
    D_a=action_dim,
    discrete=False,
)
controller = load_controller(str(controller_path))

x0_lo, x0_hi = prepare_initial_sets(
    initial_set=initial_set,
    splits_cfg=splits,
    D_total=total_dim,
)

print(f"Prepared {x0_lo.shape[0]} initial partitions with dimension {x0_lo.shape[1]}.")

Prepared 4096 initial partitions with dimension 5.


## 3. Build the reachability object

In [11]:
reach = CT_Ctl_Reach(
    rhs=rhs,
    state_dim=state_dim,
    action_dim=action_dim,
    nn_dyn=False,
    controller=controller,
    n_steps_per_control=n_steps_per_control,
    step_size=step_size,
    init_remainder=init_remainder,
    frr_rounds=frr_rounds,
    frr_stop_ratio=frr_stop_ratio,
    sr_window_size=sr_window_size,
    reference_dim=0,
)

## 4. Run reachability analysis

The first call includes JIT compilation. The second call is the steady-state run.

In [12]:
verify_fn = jax.jit(reach.verify, static_argnames=("n_total_steps",))

t0 = time.time()
ts, lowers, uppers, _, shrinked = verify_fn(x0_lo, x0_hi, n_total_steps)
jax.block_until_ready(uppers)
print(f"warmup/JIT run: {time.time() - t0:.3f}s")

t1 = time.time()
ts, lowers, uppers, _, shrinked = verify_fn(x0_lo, x0_hi, n_total_steps)
jax.block_until_ready(uppers)
print(f"compiled run: {(time.time() - t1) * 1000:.3f}ms")

warmup/JIT run: 20.573s
compiled run: 1103.615ms


## 5. Print the final per-dimension ranges

In [13]:
final_lo = jnp.min(lowers[:, -1, :], axis=0)
final_hi = jnp.max(uppers[:, -1, :], axis=0)

print(f"Final time: {float(ts[-1]):.6f}")
for name, lo, hi in zip(variable_names, final_lo, final_hi):
    print(f"{name}(T) in [{float(lo): .8e}, {float(hi): .8e}]")

Final time: 1.000000
x1(T) in [-2.00597132e-02, -1.34567059e-02]
x2(T) in [ 3.09537067e-02,  4.36638193e-02]
x3(T) in [-3.46895043e-03, -2.64071736e-03]
x4(T) in [-1.22725395e-02, -6.66928249e-03]
u1(T) in [-2.50867584e-02, -1.96394757e-02]


## Try your own system

Minimal changes are usually enough:

- Set `state_dim`, `action_dim`, `variable_names`, and the timing parameters.
- Point `dynamics_path` to a Python file with `dynamics(x)` returning the state
  derivative. The loader appends zero dynamics for the action dimensions.
- Point `controller_path` to your ONNX controller.
- You may also define dynamics and controller directly in JAX in-place without using loaders, but that is not shown in this tutorial.
- Replace `initial_set` with one `[lower, upper]` row per state/action variable.
- Increase or decrease `splits` to trade runtime for tighter ranges.